In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from statsmodels.tsa.stattools import grangercausalitytests

In [2]:
RNG_SEED = 42
np.random.seed(RNG_SEED)

In [3]:
# ---------------------------------------------------------------------------
# 1. Environments (exact reproduction of the author's classes)
# ---------------------------------------------------------------------------
class FarmIrrigationEnv(gym.Env):
    def __init__(self, crop_year_df, season_length=120, base_yield=3000.0):
        super().__init__()
        self.season_length = season_length
        self.base_yield = base_yield
        if isinstance(crop_year_df, pd.Series):
            self.data = crop_year_df.to_frame().T.reset_index(drop=True)
        else:
            self.data = crop_year_df.reset_index(drop=True)
        self.action_space = spaces.Discrete(3)
        self.observation_space = spaces.Box(low=-5, high=5, shape=(6,), dtype=np.float32)
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        idx = self.np_random.integers(0, len(self.data))
        row = self.data.iloc[idx]
        self.temp_anomaly = float(row["TempAnomaly_C"])
        self.co2_level = float(row["CO2_ppm_mean"])
        self.aerosol_level = float(row["merra2_TOTEXTTAU"])
        self.real_yield = float(row["Value"])
        self.day = 0
        self.soil_moisture = 0.5
        self.growth_stage = 0.0
        self.water_used = 0.0
        return self._get_obs(), {}

    def _get_obs(self):
        return np.array([
            self.day / self.season_length, self.soil_moisture, self.growth_stage,
            self.temp_anomaly, (self.co2_level - 400) / 50, self.aerosol_level
        ], dtype=np.float32)

    def step(self, action):
        raise NotImplementedError

In [4]:
class FarmIrrigationEnvV2(FarmIrrigationEnv):
    def step(self, action):
        base_et0 = 4.5
        et0 = base_et0 * (1 + 0.05 * max(self.temp_anomaly, 0))
        evap = et0 / 100.0
        rain_chance = max(0.1, 0.3 - 0.05 * max(self.temp_anomaly, 0))
        rain = self.np_random.binomial(1, rain_chance) * self.np_random.uniform(0.05, 0.15)
        irrigation_amount = [0.0, 0.08, 0.18][action]
        self.water_used += irrigation_amount
        self.soil_moisture += irrigation_amount + rain - evap
        self.soil_moisture = np.clip(self.soil_moisture, 0, 1)
        self.growth_stage = min(1.0, self.growth_stage + 1.0 / self.season_length)
        self.day += 1
        done = self.day >= self.season_length
        stress_penalty = -abs(self.soil_moisture - 0.6) * 5
        water_penalty = -irrigation_amount * 0.5
        reward = stress_penalty + water_penalty
        return self._get_obs(), reward, done, False, {}

In [5]:
# ---------------------------------------------------------------------------
# 2. Rebuild merged_clean_v2 exactly per the author's root-cause fix
# ---------------------------------------------------------------------------
climate_extended = pd.read_csv("data/climate_yearly_context_extended.csv")
faostat = pd.read_csv("data/faostat_yield_filtered.csv")

In [6]:
climate_full_filled = climate_extended.set_index("Year").sort_index()
climate_full_filled["merra2_TOTEXTTAU"] = climate_full_filled["merra2_TOTEXTTAU"].interpolate(
    method="linear", limit_direction="both"
)
climate_full_filled = climate_full_filled.reset_index()

In [7]:
merged_rebuilt = faostat.merge(
    climate_full_filled[["Year", "TempAnomaly_C", "CO2_ppm_mean", "merra2_TOTEXTTAU"]],
    on="Year", how="left"
)
merged_clean_v2 = merged_rebuilt.dropna(subset=["Value", "TempAnomaly_C", "CO2_ppm_mean", "merra2_TOTEXTTAU"])

In [8]:
print(f"merged_clean_v2 total rows: {len(merged_clean_v2)}")
pak_wheat_v2 = merged_clean_v2[
    (merged_clean_v2["Area"] == "Pakistan") & (merged_clean_v2["Item"] == "Wheat")
].sort_values("Year").reset_index(drop=True)
print(f"Pakistan-Wheat rows: {len(pak_wheat_v2)} (expected 22)")

merged_clean_v2 total rows: 264
Pakistan-Wheat rows: 22 (expected 22)


In [9]:
# ---------------------------------------------------------------------------
# 3. Load the real trained model
# ---------------------------------------------------------------------------
policy_v3 = PPO.load("models/ppo_farm_unified_v3.zip")
print("Model loaded. Trained timesteps:", policy_v3.num_timesteps)

Model loaded. Trained timesteps: 200160


In [10]:
# ---------------------------------------------------------------------------
# 4. Baseline comparison (reproduce Phase 5.3)
# ---------------------------------------------------------------------------
def rollout_fixed_action(env, action, seed):
    obs, _ = env.reset(seed=seed)
    total_r = 0.0
    for _ in range(120):
        obs, r, done, _, _ = env.step(action)
        total_r += r
        if done:
            break
    return total_r

In [11]:
def rollout_policy(env, policy, seed):
    obs, _ = env.reset(seed=seed)
    total_r = 0.0
    for _ in range(120):
        action, _ = policy.predict(obs, deterministic=True)
        obs, r, done, _, _ = env.step(int(action))
        total_r += r
        if done:
            break
    return total_r

In [12]:
def bootstrap_ci(data, n_boot=5000, ci=95, seed=RNG_SEED):
    rng = np.random.RandomState(seed)
    data = np.asarray(data)
    boots = [np.mean(rng.choice(data, size=len(data), replace=True)) for _ in range(n_boot)]
    lo, hi = np.percentile(boots, [(100 - ci) / 2, 100 - (100 - ci) / 2])
    return np.mean(data), lo, hi

In [13]:
conditions = {"PPO (trained)": "policy", "Never irrigate": 0, "Always light": 1, "Always heavy": 2}
baseline_rows = []
for _, row in pak_wheat_v2.iterrows():
    env = FarmIrrigationEnvV2(row)
    for label, spec in conditions.items():
        r = rollout_policy(env, policy_v3, seed=42) if spec == "policy" else rollout_fixed_action(env, spec, seed=42)
        baseline_rows.append({"Year": row["Year"], "Condition": label, "Reward": r})

In [14]:
baseline_df = pd.DataFrame(baseline_rows)
baseline_df.to_csv("results/baseline_comparison_v2.csv", index=False)

In [15]:
print(f"\nBaseline comparison ({len(pak_wheat_v2)} years, 95% bootstrap CI):")
summary_rows = []
for label in conditions:
    vals = baseline_df[baseline_df["Condition"] == label]["Reward"].values
    mean_r, lo, hi = bootstrap_ci(vals)
    summary_rows.append({"Condition": label, "mean_reward": mean_r, "ci_lo": lo, "ci_hi": hi})
    print(f"  {label:16s} mean={mean_r:7.2f}  95% CI=[{lo:7.2f}, {hi:7.2f}]")
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("results/baseline_comparison_summary.csv", index=False)


Baseline comparison (22 years, 95% bootstrap CI):
  PPO (trained)    mean= -42.96  95% CI=[ -44.69,  -41.15]
  Never irrigate   mean=-277.04  95% CI=[-282.12, -272.81]
  Always light     mean=-240.68  95% CI=[-240.79, -240.47]
  Always heavy     mean=-249.34  95% CI=[-249.34, -249.33]


In [16]:
# ---------------------------------------------------------------------------
# 5. Granger causality (reproduce Phase 5.4)
# ---------------------------------------------------------------------------
print(f"\nGranger causality (n={len(pak_wheat_v2)} years):")
granger_results = {}
for name in ["TempAnomaly_C", "CO2_ppm_mean", "merra2_TOTEXTTAU"]:
    pair = pak_wheat_v2[["Value", name]].dropna()
    res = grangercausalitytests(pair, maxlag=2, verbose=False)
    best_lag = min(res, key=lambda l: res[l][0]["ssr_ftest"][1])
    best_p = res[best_lag][0]["ssr_ftest"][1]
    sig = "significant" if best_p < 0.05 else "not significant"
    granger_results[name] = {"n": len(pair), "p": float(best_p), "lag": int(best_lag), "verdict": sig}
    print(f"  {name}: p={best_p:.4f} at lag {best_lag} -- {sig} (n={len(pair)})")


Granger causality (n=22 years):
  TempAnomaly_C: p=0.0151 at lag 1 -- significant (n=22)
  CO2_ppm_mean: p=0.0001 at lag 1 -- significant (n=22)
  merra2_TOTEXTTAU: p=0.0958 at lag 2 -- not significant (n=22)


In [17]:
# ---------------------------------------------------------------------------
# 6. Surrogate model + SHAP (reproduce Phase 5.5)
# ---------------------------------------------------------------------------
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

In [18]:
all_states, all_actions = [], []
for _, row in pak_wheat_v2.iterrows():
    env = FarmIrrigationEnvV2(row)
    obs, _ = env.reset(seed=42)
    for _ in range(120):
        action, _ = policy_v3.predict(obs, deterministic=True)
        all_states.append(obs.copy())
        all_actions.append(int(action))
        obs, _, done, _, _ = env.step(int(action))
        if done:
            break

In [19]:
states_df = pd.DataFrame(all_states, columns=[
    "day_progress", "soil_moisture", "growth_stage", "temp_anomaly", "co2_normalized", "aerosol_aod"])
states_df["action"] = all_actions

In [20]:
action_dist = states_df["action"].value_counts(normalize=True).sort_index()
print(f"\nAction distribution (n={len(pak_wheat_v2)} years):")
print(action_dist.rename({0: "no_irrigate", 1: "light", 2: "heavy"}))


Action distribution (n=22 years):
action
no_irrigate    0.729167
light          0.270833
Name: proportion, dtype: float64


In [21]:
X = states_df.drop(columns=["action"])
y = states_df["action"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y if y.nunique() > 1 else None, random_state=42)
sw = compute_sample_weight(class_weight="balanced", y=y_train)
surrogate = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
surrogate.fit(X_train, y_train, sample_weight=sw)
y_pred = surrogate.predict(X_test)
bal_acc = balanced_accuracy_score(y_test, y_pred)
print(f"Balanced accuracy (surrogate): {bal_acc:.3f}")

Balanced accuracy (surrogate): 0.948


In [22]:
explainer = shap.TreeExplainer(surrogate)
shap_values = explainer(X)
mean_shap = pd.Series(
    np.abs(shap_values.values).mean(axis=(0, 2)) if shap_values.values.ndim == 3
    else np.abs(shap_values.values).mean(axis=0),
    index=X.columns
).sort_values(ascending=False)
print("\nSHAP driver importance:")
print(mean_shap)


SHAP driver importance:
soil_moisture     4.076758
day_progress      1.705623
co2_normalized    1.187914
temp_anomaly      0.838682
aerosol_aod       0.121449
growth_stage      0.000000
dtype: float32


In [23]:
# ---------------------------------------------------------------------------
# 7. Save everything
# ---------------------------------------------------------------------------
import json
all_results = {
    "n_years": len(pak_wheat_v2),
    "baseline_comparison": summary_df.to_dict(orient="records"),
    "granger_causality": granger_results,
    "action_distribution": action_dist.rename({0: "no_irrigate", 1: "light", 2: "heavy"}).to_dict(),
    "surrogate_balanced_accuracy": float(bal_acc),
    "shap_driver_importance": mean_shap.to_dict(),
}
with open("results/results_summary.json", "w") as f:
    json.dump(all_results, f, indent=2, default=str)

In [24]:
print("\nSaved results_summary.json")


Saved results_summary.json
